# Problem W4D4: The same segmentation in OpenCV, and where the two libraries disagree

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dc00086-glitch/internship-prep/blob/main/problems/week04/w4d4_opencv_contours.ipynb)

| | |
|---|---|
| Problem ID | `W4D4` |
| Week / day | 4 / 4 |
| Topic | OpenCV: blurring, Otsu thresholding, findContours and contour areas |
| Expected time | 20-30 min |

**How to work this notebook**

1. Run the setup cell to build the data.
2. Run the **Toolkit** cell. It demonstrates every move this problem needs, on a tiny made up example, so none of it is the answer. Read its output before you start typing.
3. Fill in every `# YOUR CODE` blank, one part at a time.
4. Run the check cell as often as you like. It tells you which part is wrong and why, without giving you the answer.
5. When you are done, run the last cell and copy the tutor block.

Stuck on something more basic, like what an array even is? The [foundations notebook](https://colab.research.google.com/github/dc00086-glitch/internship-prep/blob/main/reference/foundations.ipynb) covers Python, numpy, pandas and matplotlib from the ground up, and you can keep it open in a second tab.

Nothing here needs installing. Everything used is already in Colab.


## Scenario

The imaging core's existing analysis script is written in OpenCV, because
OpenCV got there first and is very fast. You have been asked to understand it
well enough to check its numbers against your scikit-image pipeline.

The two libraries do the same job with different conventions, and the places
they differ are exactly the places bugs live. OpenCV came out of real time
computer vision, so it is built around 8 bit images, it returns things in
orders that make sense for video and not for arrays, and it describes objects
by their **outlines** rather than by which pixels belong to them.

That last difference is the interesting one. A contour is a polygon tracing
the boundary of an object, and the area of that polygon is not the same number
as the count of pixels inside it. Neither is wrong. They answer slightly
different questions, and a pipeline that mixes them produces numbers that are
consistently a few percent off with no obvious cause.

## OpenCV conventions, and what a contour actually is

**It wants 8 bit arrays.** Most OpenCV functions expect `uint8`. Hand them
floats and you get either an error or silence followed by nonsense. skimage,
by contrast, is happy with floats and often prefers them. Converting back and
forth is normal; forgetting to is a classic.

**Sizes come as (width, height).** Numpy shapes are `(rows, columns)`, which
is height then width. OpenCV kernel and size arguments are the other way
round. For a square kernel it makes no difference, which is why the mistake
survives until the day you use a non-square one.

**Kernels must be odd.** `cv2.GaussianBlur(img, (5, 5), 0)` is fine, `(4, 4)`
raises. The zero is sigma, and zero means "work it out from the kernel size".

**Thresholding returns two things.** `cv2.threshold` gives back the threshold
it used and the thresholded image, in that order. Combining
`cv2.THRESH_BINARY + cv2.THRESH_OTSU` tells it to compute the threshold by
Otsu's method and ignore the value you passed, which is why that argument is
conventionally written as 0. The output is `uint8` with values 0 and 255, not
booleans, which matters if you then try to use it as a mask.

**findContours traces outlines.** It returns a list of contours, each an array
of boundary points. `cv2.RETR_EXTERNAL` keeps only outermost contours and
ignores holes inside objects, which is usually what you want for counting
cells. `cv2.CHAIN_APPROX_SIMPLE` compresses straight runs of boundary points,
which saves memory and changes nothing about the shape.

It returns **two** values in OpenCV 4 and 5, and returned three in OpenCV 3.
Code written against the old version breaks, which is why you still find
`cv2.findContours(...)[-2]` in the wild as a version-proof way to grab the
contours whichever it is.

**Why contour area is not pixel count.** `cv2.contourArea` computes the area
of the polygon through the boundary pixel centres. The pixels on the boundary
straddle that line, so roughly half of each sits outside the polygon. The
result is consistently a little **smaller** than the pixel count, and the gap
is proportional to the perimeter, so it hurts small objects proportionally
more than big ones.

So the honest comparison is not "which is right" but "which question did I
ask". Pixel count is what skimage's `area` gives you. Polygon area is what
`contourArea` gives you. Pick one, say which, and do not mix them within a
study.

## Your task

Three parts, worked one at a time.

| Part | What you work out | Variables to fill in |
|---|---|---|
| a | blur and threshold the OpenCV way | `blurred`, `t_cv`, `binary` |
| b | the contours, and how many there are | `contours`, `n_contours` |
| c | contour areas, filtered, and compared with skimage | `areas`, `n_cells_cv`, `area_ratio` |

Below this you will find the **Setup** cell, then a **Toolkit** cell that
demonstrates every move on a tiny made up example, then **Part a**, **Part b**
and **Part c**, each with its own explanation and a cell to write in.

## Setup

Run this cell first. It builds the data with a fixed random seed, so your numbers match the checker and match mine.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from skimage import draw, measure

PROBLEM_ID = "W4D4"
TOPIC = "OpenCV: blurring, Otsu thresholding, findContours and contour areas"

IMG_SIZE = 256
MIN_AREA = 150
rng = np.random.default_rng(4401)

_img = np.full((IMG_SIZE, IMG_SIZE), 44.0)
_centres = [(45, 45), (50, 125), (42, 205), (115, 70), (125, 155), (110, 228),
            (185, 50), (180, 135), (200, 218), (240, 100), (232, 185)]
for _cy, _cx in _centres:
    _ra = float(rng.uniform(11.0, 14.0))
    _rb = float(rng.uniform(9.0, 13.0))
    _rr, _cc = draw.ellipse(_cy, _cx, _ra, _rb, shape=_img.shape,
                            rotation=float(rng.uniform(0.0, np.pi)))
    _img[_rr, _cc] += 120.0 * rng.uniform(0.8, 1.2)

# a little debris, small enough that the area filter should remove it
for _cy, _cx in [(150, 30), (30, 160), (215, 150)]:
    _rr, _cc = draw.disk((_cy, _cx), 4, shape=_img.shape)
    _img[_rr, _cc] += 110.0

_img += rng.normal(0.0, 6.0, size=_img.shape)
field = np.clip(_img, 0, 255).astype(np.uint8)

# The skimage answer, computed for you, so part c has something to compare against.
_sk_labels = measure.label(field > 110)
_sk_props = pd.DataFrame(measure.regionprops_table(_sk_labels, properties=("label", "area")))
SK_CELLS = _sk_props[_sk_props["area"] >= MIN_AREA].reset_index(drop=True)
SK_N_CELLS = int(len(SK_CELLS))
SK_TOTAL_AREA = float(SK_CELLS["area"].sum())

print("field:", field.shape, field.dtype)
print("eleven cells and three specks of debris were drawn")
print("")
print("scikit-image, for comparison later:")
print("  cells found      :", SK_N_CELLS)
print("  total pixel area : %.1f" % SK_TOTAL_AREA)

_fig, _ax = plt.subplots(figsize=(4.2, 4.2))
_ax.imshow(field, cmap="gray", vmin=0, vmax=255)
_ax.set_title("field"); _ax.axis("off")
plt.show()


## Toolkit for this problem

| What you want to do | How |
|---|---|
| blur, odd kernel only | `cv2.GaussianBlur(img, (5, 5), 0)` |
| Otsu threshold | `t, binimg = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)` |
| find outlines | `cv2.findContours(binimg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)` |
| grab the contours safely | take `[-2]` of what it returns, which works on OpenCV 3, 4 and 5 |
| the area of one contour | `cv2.contourArea(c)` |
| its perimeter | `cv2.arcLength(c, True)` |
| draw them on a copy | `cv2.drawContours(img_copy, contours, -1, 255, 1)` |

`cv2.threshold` returns a **pair**, the threshold and the image, so unpack it.
Passing `0` as the threshold value is the convention when you add
`THRESH_OTSU`, because the value is ignored and computed instead.

Run the cell below, then carry on to **Part a**, which is the next section down.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import draw, measure

# A tiny made up field. Nothing here is the answer.
demo = np.full((60, 60), 40.0)
for demo_c in [(18, 18), (40, 40)]:
    demo_rr, demo_cc = draw.disk(demo_c, 9, shape=demo.shape)
    demo[demo_rr, demo_cc] += 130.0
demo = np.clip(demo + np.random.default_rng(0).normal(0, 5, demo.shape), 0, 255).astype(np.uint8)

print("--- 1. OpenCV wants uint8 ---")
print("demo dtype:", demo.dtype)
try:
    cv2.GaussianBlur(demo.astype(float) / 255.0, (4, 4), 0)
except Exception as err:
    print("a float image with an even kernel:", type(err).__name__)
print("kernels must be odd, and sizes are given as (width, height)")

print("")
print("--- 2. threshold returns TWO things ---")
demo_blur = cv2.GaussianBlur(demo, (5, 5), 0)
demo_t, demo_bin = cv2.threshold(demo_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("returned threshold:", demo_t)
print("returned image    :", demo_bin.dtype, " unique values:", np.unique(demo_bin).tolist())
print("note it is uint8 0 and 255, not True and False")

print("")
print("--- 3. findContours, and the version trap ---")
demo_found = cv2.findContours(demo_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print("it returned", len(demo_found), "values on this OpenCV")
demo_contours = demo_found[-2]      # works whether it returned 2 or 3
print("contours found:", len(demo_contours))
print("first contour is an array of boundary points, shape:", demo_contours[0].shape)

print("")
print("--- 4. contour area is NOT pixel count ---")
demo_labels = measure.label(demo_bin > 0)
demo_px = [int((demo_labels == i).sum()) for i in range(1, demo_labels.max() + 1)]
demo_poly = sorted(round(cv2.contourArea(c), 1) for c in demo_contours)
print("skimage pixel counts :", sorted(demo_px))
print("cv2 polygon areas    :", demo_poly)
print("ratio                : %.4f" % (sum(demo_poly) / sum(demo_px)))
print("")
print("the polygon runs through the centres of the boundary pixels, so about half")
print("of each boundary pixel falls outside it. The gap scales with perimeter,")
print("which means small objects lose proportionally more than big ones.")

demo_vis = cv2.cvtColor(demo, cv2.COLOR_GRAY2BGR)
cv2.drawContours(demo_vis, demo_contours, -1, (255, 0, 0), 1)
demo_fig, demo_ax = plt.subplots(figsize=(3.4, 3.4))
demo_ax.imshow(demo_vis)
demo_ax.set_title("contours drawn on the image"); demo_ax.axis("off")
plt.show()


## Part a. Blur and threshold, the OpenCV way

Blur `field` with `cv2.GaussianBlur` using a 5 by 5 kernel and sigma 0, which
means "derive sigma from the kernel size". The kernel is passed as a tuple and
both entries must be odd.

Then threshold the blurred image with `cv2.threshold`. Pass `0` as the
threshold value, `255` as the maximum, and
`cv2.THRESH_BINARY + cv2.THRESH_OTSU` as the mode. That combination tells
OpenCV to ignore the value you gave and compute one by Otsu instead, which is
why the convention is to pass zero.

It returns **two** things: the threshold it chose and the thresholded image.
Unpack both. Forgetting that it returns a pair, and assigning the whole tuple
to one name, is the single most common OpenCV mistake.

Note what `binary` is: a `uint8` array of 0 and 255, not a boolean mask. If
you want to use it with numpy or skimage later, compare it to 0 first.

Fill in `blurred`, `t_cv`, `binary`.

Moves: `cv2.GaussianBlur(img, (5, 5), 0)`,
`t, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)`.

In [ ]:
# Part a. Blur, then threshold with Otsu, the OpenCV way.

blurred = None   # YOUR CODE: field blurred with a 5x5 Gaussian, sigma 0
t_cv = None      # YOUR CODE: the threshold cv2 chose
binary = None    # YOUR CODE: the thresholded image

print("threshold chosen:", t_cv)
print("binary dtype    :", None if binary is None else binary.dtype)
print("unique values   :", None if binary is None else np.unique(binary).tolist())


## Part b. Find the contours

`cv2.findContours` traces the outline of every object in the binary image.
Pass `cv2.RETR_EXTERNAL`, which keeps only outermost contours and ignores any
holes inside them, and `cv2.CHAIN_APPROX_SIMPLE`, which compresses straight
runs of boundary points without changing the shape.

Take the contours out of what it returns using `[-2]`. OpenCV 3 returned three
values and OpenCV 4 and 5 return two, so indexing from the end works on all of
them. This is genuinely how people write it in production code, and it is
worth recognising when you meet it.

Then count them. Compare that count against `SK_N_CELLS`, the scikit-image
answer printed in the setup cell. They may not match exactly, and either
library may be the one over-counting: the debris specks are near the area
cutoff, and the two libraries blurred and thresholded slightly differently.
Part c is where you filter by size and see whether they converge.

Fill in `contours`, `n_contours`.

Moves: `cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)`,
indexing with `[-2]`, `len()`.

In [ ]:
# Part b. Trace the outlines.

contours = None     # YOUR CODE: the contours, taken with [-2] so it works on any version
n_contours = None   # YOUR CODE: how many

print("contours found   :", n_contours)
print("skimage found    :", SK_N_CELLS, "cells after its own area filter")


## Part c. Areas, filtering, and the comparison

Compute the area of every contour with `cv2.contourArea`, as a plain list of
floats in the order the contours came.

Filter to the ones at least `MIN_AREA`, the same cutoff skimage used, and
count them. That should drop the debris specks and leave you close to the
skimage count.

Then quantify the systematic difference. Sum the areas of the contours that
passed the filter, and divide by `SK_TOTAL_AREA`, the total pixel area
scikit-image measured for its cells. Keep it as a ratio.

Think about which way that ratio should go before you look. The contour
polygon runs through the **centres** of the boundary pixels, so roughly half
of each boundary pixel lies outside it, while skimage counts those pixels in
full. So the ratio should come out a little under 1, and it should be nearer 1
for larger objects, because the boundary is a smaller share of them.

A few percent is not a bug, and it does not go away by being careful. It is
the two libraries answering slightly different questions. What matters is
picking one and not mixing them inside a study.

Fill in `areas`, `n_cells_cv`, `area_ratio`.

Moves: a list comprehension over `contours`, `cv2.contourArea`, filtering,
`sum()`, division.

In [ ]:
# Part c. Contour areas, filtered, and compared with skimage.

areas = None         # YOUR CODE: list of cv2.contourArea for every contour
n_cells_cv = None    # YOUR CODE: how many contours have area >= MIN_AREA
area_ratio = None    # YOUR CODE: total area of those contours / SK_TOTAL_AREA

print("all contour areas :", None if areas is None else sorted(round(a, 1) for a in areas))
print("cells by OpenCV   :", n_cells_cv)
print("cells by skimage  :", SK_N_CELLS)
print("area ratio cv2/skimage: %s" % area_ratio)


## Check your work

Run this cell. It prints PASS, FAIL or TODO for each part. Do not edit it.

In [ ]:
# Grading helpers. You do not need to read or edit this cell.
import base64 as _b64, json as _json, datetime as _dt, textwrap as _tw
import numpy as _np
import pandas as _pd

# Expected answers, encoded so that you do not read them by accident.
_EXP = _json.loads(_b64.b64decode("eyJhcmVhX3JhdGlvIjogMC45MjY1NDQ5NzQ3MDg1OTkxLCAiYXJlYXNfc29ydGVkIjogWzM0LjAsIDM0LjAsIDM0LjAsIDMyNi4wLCAzMjkuMCwgMzM1LjAsIDM1Ni4wLCAzODIuMCwgNDAxLjAsIDQwNS4wLCA0MDguMCwgNDE1LjAsIDQxOC4wLCA0MzguMF0sICJiaW5hcnlfdmFsdWVzIjogWzAsIDI1NV0sICJuX2JpbmFyeV9mZyI6IDQ3MDYsICJuX2NlbGxzX2N2IjogMTEsICJuX2NvbnRvdXJzIjogMTQsICJwZXJpbXNfc29ydGVkIjogWzIxLjY1NjksIDIxLjY1NjksIDIxLjY1NjksIDY4Ljc2OTYsIDY4Ljc2OTYsIDY5Ljk0MTEsIDY5Ljk0MTEsIDczLjk0MTEsIDczLjk0MTEsIDc2Ljc2OTYsIDc2Ljc2OTYsIDc2Ljc2OTYsIDc2Ljc2OTYsIDc4LjQyNjRdLCAic2hhcGUiOiBbMjU2LCAyNTZdLCAic2tfbl9jZWxscyI6IDExLCAic2tfdG90YWxfYXJlYSI6IDQ1NDcuMCwgInN1bV9hbGwiOiA0MzE1LjAsICJzdW1fYmlnIjogNDIxMy4wLCAidF9jdiI6IDk5LjB9").decode("utf-8"))

CHECK_RESULTS = []


class _Todo(Exception):
    """Raised when a part has not been attempted yet."""


def _need(name):
    g = globals()
    if name not in g:
        raise _Todo("`" + name + "` is not defined yet. Run the starter cell for this part.")
    val = g[name]
    if val is None:
        raise _Todo("`" + name + "` is still None. Fill in the # YOUR CODE blank.")
    return val


def _as_scalar(val):
    try:
        arr = _np.asarray(val, dtype=float)
    except Exception:
        return None
    if arr.size != 1:
        return None
    return float(arr.reshape(-1)[0])


def _as_array(val):
    try:
        return _np.asarray(val, dtype=float)
    except Exception:
        return None


def _num_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_scalar(a)
    if a is None:
        return False
    return bool(_np.isclose(a, float(b), rtol=rtol, atol=atol, equal_nan=True))


def _arr_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_array(a)
    if a is None:
        return False
    b = _np.asarray(b, dtype=float)
    if a.shape != b.shape:
        return False
    return bool(_np.allclose(a, b, rtol=rtol, atol=atol, equal_nan=True))


def _cols_of(obj):
    try:
        return [str(c) for c in obj.columns]
    except Exception:
        return None


def check():
    """Test every part and print PASS, FAIL or TODO for each one."""
    global CHECK_RESULTS
    names = sorted(n for n in globals() if n.startswith("_part_"))
    CHECK_RESULTS = []
    for name in names:
        label = name[len("_part_"):]
        try:
            problem = globals()[name]()
            status = "FAIL" if problem else "PASS"
            msg = problem or ""
        except _Todo as exc:
            status, msg = "TODO", str(exc)
        except Exception as exc:
            status = "FAIL"
            msg = "could not check this part, it raised %s: %s" % (type(exc).__name__, exc)
        CHECK_RESULTS.append((label, status, msg))
    _print_results()


def _print_results():
    bar = "=" * 70
    print(bar)
    print("Problem %s  |  %s" % (PROBLEM_ID, TOPIC))
    print(bar)
    for label, status, msg in CHECK_RESULTS:
        print("Part %s: %s" % (label, status))
        if msg:
            for line in _tw.wrap(msg, width=64):
                print("    " + line)
    n_pass = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
    print("-" * 70)
    print("Result: %d of %d parts passing" % (n_pass, len(CHECK_RESULTS)))
    if n_pass == len(CHECK_RESULTS):
        print("All parts pass. Run the last cell to build your tutor block.")
    print(bar)


def send_to_tutor():
    """Build the block you copy to your tutor, along with your code."""
    lines = []
    lines.append("===== SEND TO TUTOR =====")
    lines.append("Problem ID : %s" % PROBLEM_ID)
    lines.append("Topic      : %s" % TOPIC)
    lines.append("Date       : %s" % _dt.date.today().isoformat())
    lines.append("Check results:")
    if not CHECK_RESULTS:
        lines.append("  check() has not been run yet. Run the check cell above first.")
    else:
        for label, status, msg in CHECK_RESULTS:
            lines.append("  Part %s: %s" % (label, status))
            if msg:
                for line in _tw.wrap(msg, width=58):
                    lines.append("      " + line)
        n_pass = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
        lines.append("Summary: %d of %d parts passing" % (n_pass, len(CHECK_RESULTS)))
    lines.append("=========================")
    lines.append("Paste your code cells below this block.")
    return "\n".join(lines)



def _part_a():
    blurred = _need("blurred")
    t_cv = _need("t_cv")
    binary = _need("binary")

    arr = _np.asarray(blurred)
    if list(arr.shape) != _EXP["shape"]:
        return "`blurred` should be the same shape as field."
    if arr.dtype != _np.uint8:
        return ("`blurred` came back as %s. cv2.GaussianBlur on a uint8 image returns "
                "uint8; a float result means field was converted first." % arr.dtype)

    got = _as_scalar(t_cv)
    if got is None:
        if isinstance(t_cv, tuple):
            return ("`t_cv` is a tuple, so the pair cv2.threshold returns was assigned to "
                    "one name. Unpack it: t_cv, binary = cv2.threshold(...).")
        return "`t_cv` should be a single number, the first thing cv2.threshold returns."
    if not _num_close(got, _EXP["t_cv"], rtol=1e-6):
        return ("`t_cv` is not the threshold cv2 chose. Pass 0 as the value and "
                "cv2.THRESH_BINARY + cv2.THRESH_OTSU as the mode.")

    arr = _np.asarray(binary)
    if list(arr.shape) != _EXP["shape"]:
        return "`binary` should be the same shape as field."
    if arr.dtype == bool:
        return ("`binary` is boolean, but cv2.threshold returns a uint8 image of 0 and 255. "
                "Keep what cv2 gave you.")
    vals = sorted(int(v) for v in _np.unique(arr))
    if vals != _EXP["binary_values"]:
        return ("`binary` has values %s but cv2.threshold with THRESH_BINARY gives only "
                "0 and 255." % vals)
    if int((arr > 0).sum()) != _EXP["n_binary_fg"]:
        return "`binary` does not match thresholding the blurred image."
    return None


def _part_b():
    contours = _need("contours")
    n_contours = _need("n_contours")

    if isinstance(contours, tuple) and len(contours) in (2, 3) and \
            not hasattr(contours[0], "shape"):
        return ("`contours` is everything cv2.findContours returned, not the contours "
                "themselves. Take [-2] of it.")
    try:
        count = len(contours)
    except TypeError:
        return "`contours` should be a sequence of contours."
    if count != _EXP["n_contours"]:
        return ("`contours` holds %d items but %d were expected. Pass cv2.RETR_EXTERNAL "
                "and cv2.CHAIN_APPROX_SIMPLE, and run it on `binary`."
                % (count, _EXP["n_contours"]))

    got = _as_scalar(n_contours)
    if got is None:
        return "`n_contours` should be a single whole number."
    if int(round(got)) != _EXP["n_contours"]:
        return "`n_contours` is not len(contours)."
    return None


def _part_c():
    areas = _need("areas")
    n_cells_cv = _need("n_cells_cv")
    area_ratio = _need("area_ratio")

    try:
        got = sorted(round(float(a), 4) for a in areas)
    except TypeError:
        return "`areas` should be a sequence of numbers, one per contour."
    if len(got) != _EXP["n_contours"]:
        return ("`areas` has %d entries but there are %d contours, so there should be one "
                "each." % (len(got), _EXP["n_contours"]))
    if got == _EXP["perims_sorted"]:
        return ("`areas` holds perimeters, not areas. cv2.arcLength gives the perimeter and "
                "cv2.contourArea gives the area.")
    if not _arr_close(got, _EXP["areas_sorted"], rtol=1e-4, atol=1e-6):
        return "`areas` does not match cv2.contourArea over the contours."

    got = _as_scalar(n_cells_cv)
    if got is None:
        return "`n_cells_cv` should be a single whole number."
    if int(round(got)) == _EXP["n_contours"]:
        return ("`n_cells_cv` counted every contour, so the area filter was not applied. "
                "Keep only those with area at least MIN_AREA.")
    if int(round(got)) != _EXP["n_cells_cv"]:
        return "`n_cells_cv` is not the number of contours with area at least MIN_AREA."

    got = _as_scalar(area_ratio)
    if got is None:
        return "`area_ratio` should be a single number."
    if _num_close(got, _EXP["sum_big"], rtol=1e-6):
        return ("`area_ratio` is the total contour area, not a ratio. Divide by "
                "SK_TOTAL_AREA.")
    if _num_close(got, 1.0 / _EXP["area_ratio"], rtol=1e-4):
        return ("`area_ratio` is the other way up. The task asks for the OpenCV total "
                "divided by the scikit-image total, which should come out a little under 1.")
    if _num_close(got, _EXP["sum_all"] / _EXP["sk_total_area"], rtol=1e-6) and \
            not _num_close(got, _EXP["area_ratio"], rtol=1e-6):
        return ("`area_ratio` used every contour, including the debris. Sum only the ones "
                "that passed the area filter, so it matches what skimage counted.")
    if not _num_close(got, _EXP["area_ratio"], rtol=1e-4):
        return "`area_ratio` is not the filtered contour area total over SK_TOTAL_AREA."
    return None


check()


## Send to tutor

In [ ]:
# Run this last. Copy everything it prints, then paste your own code under it.
print(send_to_tutor())
